# SymBot — Symptom Checker Ingest
Merges all 4 CSV files and builds ChromaDB vectorstore.

In [1]:
# Install dependencies
import subprocess
subprocess.run(['pip', 'install', 'langchain', 'langchain-community', 'langchain-ollama',
                'chromadb', 'sentence-transformers', 'pandas'], capture_output=True)

CompletedProcess(args=['pip', 'install', 'langchain', 'langchain-community', 'langchain-ollama', 'chromadb', 'sentence-transformers', 'pandas'], returncode=0, stdout=b'Requirement already satisfied: langchain in C:\\Users\\alpha\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages (1.2.13)\r\nRequirement already satisfied: langchain-community in C:\\Users\\alpha\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages (0.4.1)\r\nRequirement already satisfied: langchain-ollama in C:\\Users\\alpha\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages (1.0.1)\r\nRequirement already satisfied: chromadb in C:\\Users\\alpha\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages (1.5.5)\r\nRequirement already satisfied: sentence-transformers in C:\\Users\\alpha\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages (5.2.0)\r\nRequirement already satisfied: pandas in C:\\Users\\alpha\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-pack

In [2]:
import pandas as pd
import os

# ── Load all 4 CSVs ──────────────────────────────────────────────
df_main    = pd.read_csv('data/dataset.csv')
df_desc    = pd.read_csv('data/symptom_Description.csv')
df_prec    = pd.read_csv('data/symptom_precaution.csv')
df_sev     = pd.read_csv('data/Symptom-severity.csv')

print('✅ All CSVs loaded')
print(f'   dataset       : {len(df_main)} rows')
print(f'   descriptions  : {len(df_desc)} rows')
print(f'   precautions   : {len(df_prec)} rows')
print(f'   severity      : {len(df_sev)} rows')

✅ All CSVs loaded
   dataset       : 4920 rows
   descriptions  : 41 rows
   precautions   : 41 rows
   severity      : 133 rows


In [3]:
# ── Build severity lookup ────────────────────────────────────────
severity_map = {}
for _, row in df_sev.iterrows():
    symptom = str(row['Symptom']).strip().replace('_', ' ').lower()
    severity_map[symptom] = int(row['weight'])

# ── Aggregate symptoms per disease ──────────────────────────────
symptom_cols = [c for c in df_main.columns if c.startswith('Symptom_')]
disease_symptoms = {}

for _, row in df_main.iterrows():
    disease = str(row['Disease']).strip()
    symptoms = []
    for col in symptom_cols:
        val = str(row[col]).strip()
        if val and val.lower() != 'nan':
            clean = val.replace('_', ' ').strip()
            if clean not in symptoms:
                symptoms.append(clean)
    if disease not in disease_symptoms:
        disease_symptoms[disease] = set()
    disease_symptoms[disease].update(symptoms)

print(f'✅ Aggregated {len(disease_symptoms)} unique diseases')

✅ Aggregated 41 unique diseases


In [4]:
# ── Build description & precaution lookups ───────────────────────
desc_map = {str(r['Disease']).strip(): str(r['Description']).strip()
            for _, r in df_desc.iterrows()}

prec_cols = [c for c in df_prec.columns if c.startswith('Precaution_')]
prec_map  = {}
for _, row in df_prec.iterrows():
    d = str(row['Disease']).strip()
    precs = [str(row[c]).strip() for c in prec_cols
             if str(row[c]).strip() and str(row[c]).strip().lower() != 'nan']
    prec_map[d] = precs

print('✅ Descriptions and precautions mapped')

✅ Descriptions and precautions mapped


In [6]:
from langchain_core.documents import Document

documents = []

for disease, symptoms in disease_symptoms.items():
    symptom_list  = sorted(symptoms)
    description   = desc_map.get(disease, 'No description available.')
    precautions   = prec_map.get(disease, [])

    # Severity score for this disease
    scores = [severity_map.get(s.lower(), 1) for s in symptom_list]
    avg_severity = round(sum(scores) / len(scores), 2) if scores else 1

    content = f"""Disease: {disease}
Symptoms: {', '.join(symptom_list)}
Description: {description}
Precautions: {', '.join(precautions) if precautions else 'Consult a doctor.'}
Average Severity Score: {avg_severity} / 7"""

    documents.append(Document(
        page_content=content,
        metadata={
            'disease': disease,
            'severity': avg_severity,
            'symptom_count': len(symptom_list)
        }
    ))

print(f'✅ Created {len(documents)} documents')
print('\nSample document:')
print(documents[0].page_content)

✅ Created 41 documents

Sample document:
Disease: Fungal infection
Symptoms: dischromic  patches, itching, nodal skin eruptions, skin rash
Description: In humans, fungal infections occur when an invading fungus takes over an area of the body and is too much for the immune system to handle. Fungi can live in the air, soil, water, and plants. There are also some fungi that live naturally in the human body. Like many microbes, there are helpful fungi and harmful fungi.
Precautions: bath twice, use detol or neem in bathing water, keep infected area dry, use clean cloths
Average Severity Score: 2.25 / 7


In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import shutil

# Clear old vectorstore if exists
if os.path.exists('vectorstore'):
    shutil.rmtree('vectorstore')

print('⏳ Loading MiniLM embeddings...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'}
)

print('⏳ Building ChromaDB vectorstore...')
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory='vectorstore'
)
vectorstore.persist()
print(f'✅ Vectorstore saved! Total documents: {vectorstore._collection.count()}')

⏳ Loading MiniLM embeddings...


C:\Users\alpha\AppData\Local\Temp\ipykernel_28716\2068958885.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(



⏳ Building ChromaDB vectorstore...
✅ Vectorstore saved! Total documents: 82


C:\Users\alpha\AppData\Local\Temp\ipykernel_28716\2068958885.py:21: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [8]:
# ── Quick test ───────────────────────────────────────────────────
results = vectorstore.similarity_search('itching skin rash fever', k=3)
print('🔍 Test query: "itching skin rash fever"')
for r in results:
    print(f'  → {r.metadata["disease"]} (severity: {r.metadata["severity"]})')

🔍 Test query: "itching skin rash fever"
  → Psoriasis (severity: 2.5)
  → Psoriasis (severity: 2.5)
  → Allergy (severity: 4.0)
